In [1]:
import os
import logging
import psycopg2

from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from datetime import datetime

# =====================================================
# ENVIRONMENT
# =====================================================

POSTGRES_HOST = os.getenv("POSTGRES_OLIST_HOST")
POSTGRES_PORT = 5432
POSTGRES_DB = os.getenv("POSTGRES_OLIST_DB")

POSTGRES_USER = os.getenv("POSTGRES_OLIST_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_OLIST_PASSWORD")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
MINIO_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

POSTGRES_JAR = "/home/jovyan/drivers/postgresql.jar"

DATA_PROC = datetime.now()

# =====================================================
# LOGGING
# =====================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

# =====================================================
# SPARK
# =====================================================

spark = (
    SparkSession.builder
    .appName("postgres_to_landing")
    .config(
        "spark.jars",
        "/home/jovyan/drivers/postgresql.jar"
    )
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        MINIO_ENDPOINT
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        MINIO_ACCESS_KEY
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        MINIO_SECRET_KEY
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

# =====================================================
# JDBC
# =====================================================

JDBC_URL = (
    f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)
# =====================================================
# LISTAR TABELAS
# =====================================================

def get_tables():

    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        database=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )

    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'raw'
        ORDER BY table_name
        """
    )

    tables = [row[0] for row in cursor.fetchall()]

    cursor.close()
    conn.close()

    return tables

# =====================================================
# EXTRAÇÃO
# =====================================================

def extract_table(table_name):

    logger.info(
        f"Extraindo raw.{table_name}"
    )

    df = (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option(
            "dbtable",
            f"raw.{table_name}"
        )
        .option("user", POSTGRES_USER)
        .option("password", POSTGRES_PASSWORD)
        .option(
            "driver",
            "org.postgresql.Driver"
        )
        .load()
    )

    # Adiciona timestamp de processamento
    df = df.withColumn(
        "dataproc",
        lit(DATA_PROC)
    )

    destination = (
        f"s3a://landing/olist/{table_name}"
    )

    logger.info(
        f"Gravando {destination}"
    )

    (
        df.write
        .mode("overwrite")
        .parquet(destination)
    )

    logger.info(
        f"{table_name} finalizada"
    )

# =====================================================
# MAIN
# =====================================================

def main():

    tables = get_tables()

    logger.info(
        f"{len(tables)} tabelas encontradas"
    )

    for table in tables:

        try:

            extract_table(table)

        except Exception as e:

            logger.exception(
                f"Erro em {table}: {e}"
            )

    logger.info(
        "Processamento concluído"
    )

# =====================================================
# ENTRY POINT
# =====================================================

if __name__ == "__main__":
    main()

2026-06-14 19:47:22,458 - INFO - 8 tabelas encontradas
2026-06-14 19:47:22,459 - INFO - Extraindo raw.customers
2026-06-14 19:47:23,637 - INFO - Gravando s3a://landing/olist/customers
2026-06-14 19:47:26,182 - INFO - customers finalizada
2026-06-14 19:47:26,183 - INFO - Extraindo raw.geolocation
2026-06-14 19:47:26,221 - INFO - Gravando s3a://landing/olist/geolocation
2026-06-14 19:47:28,316 - INFO - geolocation finalizada
2026-06-14 19:47:28,317 - INFO - Extraindo raw.order_items
2026-06-14 19:47:28,348 - INFO - Gravando s3a://landing/olist/order_items
2026-06-14 19:47:29,341 - INFO - order_items finalizada
2026-06-14 19:47:29,342 - INFO - Extraindo raw.order_payments
2026-06-14 19:47:29,372 - INFO - Gravando s3a://landing/olist/order_payments
2026-06-14 19:47:29,881 - INFO - order_payments finalizada
2026-06-14 19:47:29,882 - INFO - Extraindo raw.order_reviews
2026-06-14 19:47:29,910 - INFO - Gravando s3a://landing/olist/order_reviews
2026-06-14 19:47:30,667 - INFO - order_reviews fi

In [2]:
spark.stop()